# Síntesis completa (OE1 y OE3)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ninguna")
assert torch.cuda.is_available(), "la generación y el entrenamiento necesitan GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess

from google.colab import userdata

REPO = "github.com/Riv2oo4/Trabajo-De-Graduacion.git"
RAMA = "oe4-oe5-y-reproducibilidad"

token = userdata.get("GITHUB_TOKEN")

# si una corrida anterior dejó el proceso parado dentro de /content/repo,
# borrarlo deja getcwd sin directorio y ningún subproceso vuelve a arrancar
os.chdir("/content")

!rm -rf /content/repo
subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA,
                f"https://x-access-token:{token}@{REPO}", "/content/repo"], check=True)

assert os.path.exists("/content/repo/analisis/corpus_sintetico.py"), \
    f"el clon de {RAMA} no trae corpus_sintetico.py: falta empujar los cambios al remoto"

os.chdir("/content/repo/analisis")
import sys
sys.path.insert(0, "/content/repo/analisis")

# los módulos ya importados en un kernel viejo no se recargan solos
for m in [k for k in sys.modules if k in (
        "corpus_sintetico", "dataset_sintesis", "experimento_sintesis",
        "inferencia", "modelo", "evaluacion_ciega", "entrenamiento")]:
    del sys.modules[m]

## Dependencias

In [ ]:
!pip install -q "diffusers>=0.31" accelerate safetensors
import diffusers
print("diffusers", diffusers.__version__)

## Datos

In [ ]:
DRIVE = "/content/drive/MyDrive/Corpus-tesis"
ZIP_REALES = f"{DRIVE}/corpus_real.zip"
MANIPULADAS = f"{DRIVE}/manipuladas"
RESULTADOS = f"{DRIVE}/resultados"
SINTETICAS = f"{DRIVE}/sinteticas"

assert os.path.exists(ZIP_REALES), ZIP_REALES
assert os.path.isdir(MANIPULADAS), MANIPULADAS
assert os.path.isdir(RESULTADOS), RESULTADOS
os.makedirs(SINTETICAS, exist_ok=True)

In [ ]:
!mkdir -p /content/reales && unzip -q -o "{ZIP_REALES}" -d /content/reales

REALES = "/content/reales/real"
CATALOGO = "/content/repo/corpus/catalogo.csv"
METADATA = "/content/repo/corpus/metadata_real.csv"
print(len(os.listdir(REALES)), "reales")

## Generación del corpus sintético

In [ ]:
# unas tres horas de GPU para 300 imágenes; si la sesión se corta, volver a
# correr la celda reanuda y no regenera lo que ya está en disco
CMD = (
    "python corpus_sintetico.py"
    f" --raiz-real {REALES}"
    f" --catalogo {CATALOGO}"
    f" --metadata {METADATA}"
    f" --salida '{SINTETICAS}'"
    " --cantidad 200"
    " --libres 100"
)
!{CMD}

In [ ]:
import pandas as pd

CATALOGO_SINTETICO = f"{DRIVE}/catalogo_sintetico.csv"
sint = pd.read_csv(CATALOGO_SINTETICO)
print(len(sint), "imágenes sintéticas")
sint.groupby(["condicion", "particion"]).size().unstack(fill_value=0)

In [ ]:
# ninguna sintética puede compartir fotografía de origen con otra partición
pareadas = sint[sint.condicion.eq("pareada")]
cat = pd.read_csv(CATALOGO).set_index("archivo")
mal = [r.archivo for r in pareadas.itertuples()
       if cat.loc[r.ref, "particion"] != r.particion]
assert not mal, mal
print("particiones heredadas correctamente")

## Entrenamiento y matriz del espectro

In [ ]:
TABLA_OE3 = f"{RESULTADOS}/detector_oe1.csv"
PUNTO_LOCAL = f"{RESULTADOS}/puntos_de_control/imagenet_ajustado.pt"

from inferencia import umbral_de_resultados
UMBRAL_LOCAL = umbral_de_resultados(TABLA_OE3)
print("corte del detector local:", UMBRAL_LOCAL)

In [ ]:
CMD = (
    "python experimento_sintesis.py"
    f" --catalogo {CATALOGO}"
    f" --catalogo-sintetico '{CATALOGO_SINTETICO}'"
    f" --reales {REALES}"
    f" --manipuladas '{MANIPULADAS}'"
    f" --sinteticas '{SINTETICAS}'"
    f" --pesos-local '{PUNTO_LOCAL}'"
    f" --umbral-local {UMBRAL_LOCAL}"
    f" --salida '{RESULTADOS}/sintesis.csv'"
    f" --salida-matriz '{RESULTADOS}/matriz_espectro.csv'"
    " --epocas 10"
)
!{CMD}

In [ ]:
sin = pd.read_csv(f"{RESULTADOS}/sintesis.csv")
sin[["agregacion", "condicion", "exactitud_balanceada", "auc", "f1",
     "umbral", "sentido", "n_imagenes"]]

In [ ]:
matriz = pd.read_csv(f"{RESULTADOS}/matriz_espectro.csv")
matriz.pivot(index="clase", columns="clasificador",
             values=["puntaje_medio", "tasa_disparo"])

In [ ]:
# el resultado que la matriz existe para medir: cada cabeza responde a lo suyo
m = matriz.set_index(["clasificador", "clase"])
for cab, propia in (("local", "manipulada"), ("sintesis", "sintetica_pareada")):
    ajena = "sintetica_pareada" if cab == "local" else "manipulada"
    print(f"{cab}: con {propia} {m.loc[(cab, propia), 'tasa_disparo']:.3f}"
          f" [{m.loc[(cab, propia), 'tasa_disparo_ic_bajo']:.3f},"
          f" {m.loc[(cab, propia), 'tasa_disparo_ic_alto']:.3f}]"
          f" | con {ajena} {m.loc[(cab, ajena), 'tasa_disparo']:.3f}"
          f" [{m.loc[(cab, ajena), 'tasa_disparo_ic_bajo']:.3f},"
          f" {m.loc[(cab, ajena), 'tasa_disparo_ic_alto']:.3f}]")

In [ ]:
# el punto de control queda junto a su tabla, que ya está en Drive
PUNTO_SINTESIS = f"{RESULTADOS}/sintesis.pt"
assert os.path.exists(PUNTO_SINTESIS), PUNTO_SINTESIS
print(round(os.path.getsize(PUNTO_SINTESIS) / 1e6, 1), "MB")

## Prototipo con las dos cabezas

In [ ]:
!pip install -q "transformers>=4.49" gradio

for m in [k for k in sys.modules if k.startswith(("app", "sistema", "dobles"))]:
    del sys.modules[m]

os.chdir("/content/repo/despliegue")

CMD = (
    "python app.py --publico"
    f" --punto-de-control '{PUNTO_LOCAL}'"
    f" --resultados '{TABLA_OE3}'"
    f" --punto-de-control-sintesis '{PUNTO_SINTESIS}'"
    f" --resultados-sintesis '{RESULTADOS}/sintesis.csv'"
)
!{CMD}